# Experiment 009: Llama 3.2-1B-Instruct — Smoke Test (250 steps)

Quick pipeline validation before a full run.

| Parameter | Value |
|-----------|-------|
| Model | meta-llama/Llama-3.2-1B-Instruct (1.24B params) |
| Data | ~5K (v2 generator) |
| Steps | 250 |
| Effective batch | 16 (2 x 8) |
| Epochs | ~1.0 |
| Precision | bf16 (L4 native) |

**Runtime:** L4 GPU (24GB). ~10 min.

**Success criteria:** Non-NaN eval loss, model outputs routing tokens.

**Learning from 009a:** fp16 on T4 causes NaN/weight corruption. Always use bf16 on L4.

In [ ]:
!pip install -q transformers datasets accelerate torch tensorboard bitsandbytes

In [ ]:
import torch, os, subprocess, math, json, numpy as np
assert torch.cuda.is_available(), 'No GPU!'
assert torch.cuda.is_bf16_supported(), 'Need bf16 - use L4 runtime, not T4'
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
from huggingface_hub import login
try:
    from google.colab import userdata
    login(token=userdata.get('HF_TOKEN'))
    print('Authenticated')
except Exception:
    login()

## 1. Generate Data (5K for smoke test)

In [ ]:
if not os.path.exists('tagzeit-gemma-2b'):
    !git clone https://github.com/mgosal/tagzeit-gemma-2b.git
!cd tagzeit-gemma-2b && npm install --silent

!cd tagzeit-gemma-2b && python tools/extract_hard_negatives.py \
    --output data/hard_negatives/complex_tempqa_noroute.jsonl --count 7000

!cd tagzeit-gemma-2b && node experiments/temporal-tagzeit/src/generator_route.js \
    --count 5000 \
    --output data/train/train_routed_009.jsonl \
    --eval data/eval/eval_routed_009.jsonl \
    --hard-negatives data/hard_negatives/complex_tempqa_noroute.jsonl

!cd tagzeit-gemma-2b && python tools/format_for_model.py \
    --model_id meta-llama/Llama-3.2-1B-Instruct \
    --input data/train/train_routed_009.jsonl \
    --output data/train/train_routed_009_llama1b.jsonl \
    --eval_input data/eval/eval_routed_009.jsonl \
    --eval_output data/eval/eval_routed_009_llama1b.jsonl

TRAIN_FILE = 'tagzeit-gemma-2b/data/train/train_routed_009_llama1b.jsonl'
EVAL_FILE  = 'tagzeit-gemma-2b/data/eval/eval_routed_009_llama1b.jsonl'

for f in [TRAIN_FILE, EVAL_FILE]:
    n = int(subprocess.check_output(['wc', '-l', f]).split()[0])
    print(f'{os.path.basename(f)}: {n} samples')

with open(TRAIN_FILE) as fh:
    sample = json.loads(fh.readline())
    print(f'\nSample:\n{sample["text"][:300]}')

## 2. Load Model + Domain Tokens

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = 'meta-llama/Llama-3.2-1B-Instruct'
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16, device_map='auto')

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    model.config.pad_token_id = tokenizer.eos_token_id

ROUTE_TOKENS = [
    '[ROUTE]', '[/ROUTE]', '[NO_ROUTE]',
    '[ROUTE_TIME_ADD]', '[ROUTE_TIME_SUB]', '[ROUTE_DURATION_BETWEEN]',
    '[ROUTE_CALENDAR_SHIFT]', '[ROUTE_TIMEZONE_CONVERT]',
    '[HEAD_TIME]', '[HEAD_DURATION]', '[HEAD_DATE]',
    '[REF_1]', '[REF_2]', '[REF_3]',
]
ROUTE_TOKENS += [f'[ARG_HOUR_{str(h).zfill(2)}]' for h in range(24)]
ROUTE_TOKENS += [f'[ARG_MIN_{str(m).zfill(2)}]' for m in range(60)]
ROUTE_TOKENS += ['[ARG_MON]', '[ARG_TUE]', '[ARG_WED]', '[ARG_THU]', '[ARG_FRI]', '[ARG_SAT]', '[ARG_SUN]']

num_added = tokenizer.add_tokens(ROUTE_TOKENS)
model.resize_token_embeddings(len(tokenizer))

embed_dim = model.config.hidden_size
with torch.no_grad():
    new_start = len(tokenizer) - num_added
    for i in range(num_added):
        freq = np.array([1.0 / (10000 ** (2 * j / embed_dim)) for j in range(embed_dim)])
        phase = (i + 1) * freq
        init_vec = np.sin(phase) * 0.02
        model.get_input_embeddings().weight[new_start + i] = torch.tensor(init_vec, dtype=torch.bfloat16)

print(f'Loaded: {sum(p.numel() for p in model.parameters()):,} params, {num_added} domain tokens')
print(f'VRAM: {torch.cuda.memory_allocated() / 1e9:.2f} GB')

## 3. Tokenize + Train

In [ ]:
from datasets import load_dataset
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

dataset = load_dataset('json', data_files={'train': TRAIN_FILE, 'eval': EVAL_FILE})

def tokenize_fn(examples):
    result = tokenizer(examples['text'], truncation=True, max_length=384, padding='max_length')
    result['labels'] = result['input_ids'].copy()
    return result

tokenized = dataset.map(tokenize_fn, batched=True, remove_columns=['text'])
tokenized.set_format('torch')

n_train = len(tokenized['train'])
print(f'Train: {n_train}, Eval: {len(tokenized["eval"])}')
print(f'250 steps x 16 batch = {250*16:,} samples = {(250*16)/n_train:.1f} epochs')

In [ ]:
OUTPUT_DIR = '/content/tagzeit-009-smoke'

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    max_steps=250,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=5e-5,
    lr_scheduler_type='cosine',
    warmup_steps=50,
    weight_decay=0.01,
    optim='adamw_8bit',
    bf16=True,
    fp16=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    eval_strategy='steps',
    eval_steps=125,
    save_strategy='no',
    logging_steps=25,
    logging_dir=f'{OUTPUT_DIR}/logs',
    report_to='tensorboard',
    seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized['train'],
    eval_dataset=tokenized['eval'],
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
    processing_class=tokenizer,
)

print(f'Ready: 250 steps, bf16, VRAM {torch.cuda.memory_allocated()/1e9:.1f}/{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

In [ ]:
train_result = trainer.train()

print(f'\nTrain loss: {train_result.training_loss:.4f}')
print(f'Runtime: {train_result.metrics["train_runtime"]:.0f}s')

eval_results = trainer.evaluate()
eval_loss = eval_results['eval_loss']
print(f'Eval loss: {eval_loss:.4f}')
print(f'Eval perplexity: {math.exp(eval_loss):.2f}')

if math.isnan(eval_loss):
    print('\nFAIL: eval loss is NaN - weights corrupted')
elif eval_loss > 10:
    print(f'\nWARNING: eval loss very high ({eval_loss:.1f})')
else:
    print(f'\nPASS: eval loss finite and reasonable')

## 4. Inference Test

In [ ]:
test_prompts = [
    ('What time is it 1 minute after 23:59?', 'ADD'),
    ('What time was it 30 minutes before 00:15?', 'SUB'),
    ('How much time is there between 09:00 and 17:30?', 'BETWEEN'),
    ('The meeting starts at 14:30 and lasts 45 minutes. When does it end?', 'ADD'),
    ('What is 42 + 18?', 'NO_ROUTE'),
    ('When was The Secret Garden published?', 'NO_ROUTE'),
]

check_map = {
    'ADD': '[ROUTE_TIME_ADD]',
    'SUB': '[ROUTE_TIME_SUB]',
    'BETWEEN': '[ROUTE_DURATION_BETWEEN]',
    'NO_ROUTE': '[NO_ROUTE]',
}

model.eval()
correct = 0

for prompt, expected in test_prompts:
    messages = [{'role': 'user', 'content': prompt}]
    input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(input_text, return_tensors='pt').to(model.device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=64, do_sample=False)
    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=False)
    response = response.split('<|eot_id|>')[0].strip()

    got_right = check_map[expected] in response
    if got_right:
        correct += 1

    print(f"{'PASS' if got_right else 'FAIL'} [{expected}] {prompt}")
    print(f'   {response[:150]}')
    print()

print('=' * 50)
print(f'Routing: {correct}/{len(test_prompts)} ({correct/len(test_prompts)*100:.0f}%)')
print()

# Check model is alive
if all(c == '<' for c in response[:5]):
    print('WARNING: Model may be outputting only special tokens (EOS spam)')
else:
    print('Model is producing non-trivial output')

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {OUTPUT_DIR}/logs

---

## Next: Full Run

If smoke test passes (non-NaN loss, routing tokens in output), scale up:
- Regenerate 100K data: change `--count 5000` to `--count 100000`
- Set `max_steps=5000`
- Mount Google Drive for checkpoint persistence
- Set `save_strategy='steps'`, `save_steps=1000`